In [488]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import roc_auc_score

In [489]:
# Loading the treated data
train_dataset = pd.read_csv('data/train_dataset_treated.csv')

# Loading the control dataset
control_dataset = pd.read_csv('data/train_radiomics_occipital_CONTROL.csv')

In [490]:
# Guaranteeing the same columns in both datasets
control_dataset = control_dataset[train_dataset.columns]

In [491]:
# --- Using encoding to transform the categorical columns into numerical columns
replace_map = {'Transition': {'CN-CN': 0, 'AD-AD': 1, 'CN-MCI': 2, 'MCI-AD': 3, 'MCI-MCI': 4}}
control_dataset.replace(replace_map, inplace=True)
control_dataset.replace(replace_map, inplace=True)

C:\Users\Utilizador\AppData\Local\Temp\ipykernel_17808\1480064811.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  control_dataset.replace(replace_map, inplace=True)


In [492]:
# Getting the number of rows for each target value
print('Number of rows for each target value in the control dataset')
print(control_dataset['Transition'].value_counts())


Number of rows for each target value in the control dataset
Transition
0    96
4    71
3    68
1    60
2    10
Name: count, dtype: int64


In [493]:
# Defining the function that will use the model to complete the missing values
# Using the model to complete the test dataset

def complete_dataset(model_name, model):

    test_dataset = pd.read_csv('data/test_dataset_treated.csv')

    test_pred = model.predict(test_dataset)
    test_dataset['Transition'] = test_pred
    test_dataset.head()

    # Dropping all columns but the Transition column
    test_dataset.drop(test_dataset.columns.difference(['Transition']), axis=1, inplace=True)

    # Creating a RowId column to store the index, starting from 1
    test_dataset['RowId'] = np.arange(1, test_dataset.shape[0] + 1)

    # Placing the RowId column in the first position
    cols = test_dataset.columns.tolist()
    cols = cols[-1:] + cols[:-1]
    test_dataset = test_dataset[cols]

    # Transforming the Transition column back to its original values
    replace_map = {'Transition': {0: 'CN-CN', 1: 'AD-AD', 2: 'CN-MCI', 3: 'MCI-AD', 4: 'MCI-MCI'}}
    test_dataset.replace(replace_map, inplace=True)
    test_dataset.head()

    # Saving the test dataset to a csv file
    test_dataset.to_csv('test_predictions_' + model_name + '.csv', index=False)

## Random Forest

In [ ]:
# Running a Random Forest Classifier
#model_name = 'random_forest'
X = train_dataset.drop('Transition', axis=1)
y = train_dataset['Transition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

rfc = RandomForestClassifier(n_estimators=125, random_state=123)
rfc.fit(X_train, y_train)
rfc_pred = rfc.predict(X_test)

# Printing f1 score
print("f1_score for the test data: ")
print(f1_score(y_test, rfc_pred, average='weighted'))


# Running for the control dataset as well
X_control = control_dataset.drop('Transition', axis=1)
y_control = control_dataset['Transition']

control_pred = rfc.predict(X_control)
print("\nf1_score for the control data (expected around 0.2): ")
print(f1_score(y_control, control_pred, average='weighted'))

# Calculating balanced accuracy for the control dataset (this should result in around ?)
print("Balanced accuracy for the control data (expected around ?): ")
print(balanced_accuracy_score(y_control, control_pred))
# DUVIDA -> PERCEBER SE ESTE VALOR DEVE SER 0.5 OU 0.2 (no caso é o da balanced accuracy)
# DUVIDA 2 -> PERCEBER quais as melhores métricas para garantir que os resultados locais vão ser parecidos com os resultados do Kaggle
# DUVIDA 3 -> PERCEBER se oversampling faz sentido -> nós experimentamos e os resultados locais são bons mas no kaggle são péssimos

# Calculating the AUROC for the control dataset
print("AUROC for the control data (expected around 0.5): ")
control_pred_proba = rfc.predict_proba(X_control)
print(roc_auc_score(y_control, control_pred_proba, multi_class='ovr'))


f1_score for the test data: 
0.5383038494666402

f1_score for the control data (expected around 0.2): 
0.20836223025998174
Balanced accuracy for the control data (expected around ?): 
0.2063849765258216
AUROC for the control data (expected around 0.5): 
0.5656001735291601


In [495]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_test, rfc_pred))

Confusion matrix
[[13  0  3  1  2]
 [ 0  9  1  5  0]
 [ 0  0 17  1  0]
 [ 3  6  0  4  3]
 [ 8  4  0  3  7]]


In [496]:
# Printing the classification report
print("Classification report")
print(classification_report(y_test, rfc_pred))

Classification report
              precision    recall  f1-score   support

           0       0.54      0.68      0.60        19
           1       0.47      0.60      0.53        15
           2       0.81      0.94      0.87        18
           3       0.29      0.25      0.27        16
           4       0.58      0.32      0.41        22

    accuracy                           0.56        90
   macro avg       0.54      0.56      0.54        90
weighted avg       0.55      0.56      0.54        90



In [497]:
# Checking classification report for the control dataset
print("Classification report for the control dataset")
print(classification_report(y_control, control_pred))

Classification report for the control dataset
              precision    recall  f1-score   support

           0       0.33      0.83      0.47        96
           1       0.46      0.10      0.16        60
           2       0.00      0.00      0.00        10
           3       0.00      0.00      0.00        68
           4       0.16      0.10      0.12        71

    accuracy                           0.30       305
   macro avg       0.19      0.21      0.15       305
weighted avg       0.23      0.30      0.21       305



In [498]:
# Completing the test dataset
complete_dataset('random_forest', rfc)

## XGBoost

In [499]:
#model_name = 'xgboost'
X = train_dataset.drop('Transition', axis=1)
y = train_dataset['Transition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=123)

xgbc = xgb.XGBClassifier(max_depth = 1, objective='multi:softprob', random_state=123, learning_rate=0.2, n_estimators=50, num_class=5)
xgbc.fit(X_train, y_train)
xgbc_pred = xgbc.predict(X_test)

# Printing f1 score
print("f1_score for the test data: ")
print(f1_score(y_test, xgbc_pred, average='weighted'))

# Running for the control dataset as well
X_control = control_dataset.drop('Transition', axis=1)
y_control = control_dataset['Transition']

control_pred = xgbc.predict(X_control)
print("\nf1_score for the control data (expected around 0.2): ")
print(f1_score(y_control, control_pred, average='weighted'))

# Calculating balanced accuracy for the control dataset (this should result in around ?)
print("Balanced accuracy for the control data (expected around ?): ")
print(balanced_accuracy_score(y_control, control_pred))

# Calculating the AUROC for the control dataset
control_pred_proba = xgbc.predict_proba(X_control)
print("AUROC for the control data (expected around 0.5): ")
print(roc_auc_score(y_control, control_pred_proba, multi_class='ovr'))


f1_score for the test data: 
0.587756359300477

f1_score for the control data (expected around 0.2): 
0.19441077167148027
Balanced accuracy for the control data (expected around ?): 
0.22134803921568627
AUROC for the control data (expected around 0.5): 
0.6005907041876133


In [500]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_test, xgbc_pred))

Confusion matrix
[[12  0  2  3  2]
 [ 0  9  0  6  0]
 [ 0  0 17  1  0]
 [ 4  5  0  6  1]
 [ 2  3  3  5  9]]


In [501]:
# Printing the classification report
print("Classification report")
print(classification_report(y_test, xgbc_pred))

Classification report
              precision    recall  f1-score   support

           0       0.67      0.63      0.65        19
           1       0.53      0.60      0.56        15
           2       0.77      0.94      0.85        18
           3       0.29      0.38      0.32        16
           4       0.75      0.41      0.53        22

    accuracy                           0.59        90
   macro avg       0.60      0.59      0.58        90
weighted avg       0.62      0.59      0.59        90



In [502]:
# Completing the test dataset
complete_dataset('xgboost', xgbc)